In [1]:
import os
import re
import yaml
import pandas as pd

In [2]:
def summarize_combo_results(experiment_dir):
    rows = []

    yaml_dir = os.path.join(experiment_dir, "yaml")
    result_dir = os.path.join(experiment_dir, "result")

    for yaml_name in sorted(os.listdir(yaml_dir)):
        match = re.match(r"combo_(\d+)\.yaml", yaml_name)

        if not match:
            continue

        combo = int(match.group(1))
        yaml_path = os.path.join(yaml_dir, yaml_name)

        with open(yaml_path, "r") as f:
            config = yaml.safe_load(f)

        ratios = config["prune_ratios"]

        for method in ["omp", "imp"]:
            result_path = os.path.join(
                result_dir,
                f"combo_{combo}_{method}.txt"
            )

            if not os.path.exists(result_path):
                continue

            with open(result_path, "r") as f:
                result = f.read()

            accuracy_match = re.search(r"\(([\d.]+)%\)", result)

            if not accuracy_match:
                continue

            accuracy = float(accuracy_match.group(1))

            row = {
                "Combo": combo,
                "Method": method.upper()
            }

            for layer_name, ratio in ratios.items():
                feature = layer_name.split(".")[1]
                row[f"Feature {feature}"] = int(ratio * 100)

            row["Accuracy"] = accuracy
            rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(["Combo", "Method"]).reset_index(drop=True)

    return df

In [3]:
def get_top_3_combos(df):
    top_results = []

    for method in ["OMP", "IMP"]:
        method_df = df[df["Method"] == method]
        top_3 = method_df.nlargest(3, "Accuracy").copy()
        top_3["Rank"] = range(1, len(top_3) + 1)
        top_results.append(top_3)

    return pd.concat(top_results, ignore_index=True)

In [4]:
unstructured_df = summarize_combo_results("combo_experiment/unstructured")
unstructured_df

,Combo,Method,Feature 0,Feature 3,Feature 7,Feature 10,Feature 14,Feature 17,Feature 21,Feature 24,Feature 28,Feature 31,Accuracy
0,1,IMP,0,80,80,80,80,80,80,80,80,80,93.61
1,1,OMP,0,80,80,80,80,80,80,80,80,80,93.55
2,2,IMP,0,70,70,80,70,70,90,90,60,90,93.58
3,2,OMP,0,70,70,80,70,70,90,90,60,90,93.72
4,3,IMP,0,70,70,80,70,70,85,90,65,90,93.73
5,3,OMP,0,70,70,80,70,70,85,90,65,90,93.77
6,4,IMP,0,70,70,80,70,70,80,90,70,90,93.74
7,4,OMP,0,70,70,80,70,70,80,90,70,90,94.02
8,5,IMP,0,70,70,80,70,70,85,85,70,90,93.47
9,5,OMP,0,70,70,80,70,70,85,85,70,90,93.69


In [5]:
unstructured_top3 = get_top_3_combos(unstructured_df)
unstructured_top3

,Combo,Method,Feature 0,Feature 3,Feature 7,Feature 10,Feature 14,Feature 17,Feature 21,Feature 24,Feature 28,Feature 31,Accuracy,Rank
0,4,OMP,0,70,70,80,70,70,80,90,70,90,94.02,1
1,6,OMP,0,70,70,80,70,70,85,90,70,85,93.91,2
2,9,OMP,0,70,70,90,60,70,90,90,65,90,93.89,3
3,8,IMP,0,70,80,80,60,70,90,90,65,90,93.89,1
4,4,IMP,0,70,70,80,70,70,80,90,70,90,93.74,2
5,3,IMP,0,70,70,80,70,70,85,90,65,90,93.73,3


In [6]:
filter_df = summarize_combo_results("combo_experiment/filter")
filter_df

,Combo,Method,Feature 0,Feature 3,Feature 7,Feature 10,Feature 14,Feature 17,Feature 21,Feature 24,Feature 28,Feature 31,Accuracy
0,1,IMP,0,40,40,40,40,40,40,40,40,40,92.76
1,1,OMP,0,40,40,40,40,40,40,40,40,40,93.00
2,2,IMP,0,30,30,30,35,35,40,45,45,45,93.34
3,2,OMP,0,30,30,30,35,35,40,45,45,45,93.38
4,3,IMP,0,30,35,35,35,40,40,45,45,45,93.32
5,3,OMP,0,30,35,35,35,40,40,45,45,45,93.19
6,4,IMP,0,25,30,35,35,40,45,45,45,45,93.51
7,4,OMP,0,25,30,35,35,40,45,45,45,45,93.31
8,5,IMP,0,25,30,30,35,40,45,45,50,45,93.31
9,5,OMP,0,25,30,30,35,40,45,45,50,45,93.43


In [7]:
filter_top3 = get_top_3_combos(filter_df)
filter_top3

,Combo,Method,Feature 0,Feature 3,Feature 7,Feature 10,Feature 14,Feature 17,Feature 21,Feature 24,Feature 28,Feature 31,Accuracy,Rank
0,5,OMP,0,25,30,30,35,40,45,45,50,45,93.43,1
1,2,OMP,0,30,30,30,35,35,40,45,45,45,93.38,2
2,4,OMP,0,25,30,35,35,40,45,45,45,45,93.31,3
3,4,IMP,0,25,30,35,35,40,45,45,45,45,93.51,1
4,8,IMP,0,30,25,35,35,40,45,45,50,45,93.35,2
5,2,IMP,0,30,30,30,35,35,40,45,45,45,93.34,3
